## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para disponer de persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Bajar datasets si hace falta

In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp "/content/buckets/b1/kaggle/kaggle (2).json"  ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Modelo AutoARIMA

## 1.1 Init Experimento

In [3]:
# instalacion de paquetes que NO vienen por default en Colab
!pip install uv
!uv pip install -q kaggle
!uv pip install -q pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 62.7 MB/s eta 0:00:00


In [4]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [5]:
import os as os
import numpy as np
import polars as pl
from pmdarima import auto_arima

import warnings
warnings.filterwarnings("ignore")

Por favor, cargar aqui SU semilla primigenia

In [6]:
# defino los parametros
PARAM = {'experimento':'AA01',
  'kaggle_competition':'labo-iii-2026-ba',
  'semilla_primigenia':199410
}

In [7]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/AA01


## 1.2 Init AutoARIMA

In [8]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [9]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [10]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


In [11]:
display(tb_ventas)

product_id,periodo,tn
i64,i64,f64
20001,201701,934.77222
20001,201702,798.0162
20001,201703,1303.35771
20001,201704,1069.9613
20001,201705,1502.20132
…,…,…
21276,201908,0.01265
21276,201909,0.01856
21276,201910,0.02079


Opcion de empiojar el dataset, agregando ruido relativo a las ventas
<br>Un Experimento no se le niega a nadie

In [12]:
empiojar=True
empiojar_ruido=0.4

if empiojar:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=empiojar_ruido, size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


In [13]:
display(tb_ventas)

product_id,periodo,tn
i64,i64,f64
20001,201701,763.804602
20001,201702,909.379161
20001,201703,1762.979824
20001,201704,552.831425
20001,201705,2062.832845
…,…,…
21276,201908,0.0178
21276,201909,0.028125
21276,201910,0.023041


In [14]:
# donde voy a guardar el resultado final
arch_tb_final = 'tb_final.csv'

try:
    # Attempt to load the existing file
    tb_final = pl.read_csv(arch_tb_final)
except FileNotFoundError:
    # donde voy a guardar el resultado final
    tb_final = tb_apredecir.clone()
    tb_final = tb_final.with_columns(pl.lit(None).cast(pl.Float64).alias('tn'))


tb_final = tb_final.sort(["product_id"])


# cuento cuantos registros NO puedieron calcularse
# en mi corrida esto da  209

tb_final["tn"].is_null().sum()


0

In [15]:
correrdeCero=True

if correrdeCero:
  tb_final = tb_apredecir.clone()
  tb_final = tb_final.with_columns(pl.lit(None).cast(pl.Float64).alias('tn'))


tb_final = tb_final.sort(["product_id"])

In [16]:
display(tb_final)

product_id,tn
i64,f64
20001,null
20002,null
20003,null
20004,null
20005,null
…,…
21263,null
21265,null
21266,null


## 1.3 Primera pasada AutoARIMA

Tengo en cuenta la estacionalidad de 12 periodos
<br> muchos van a quedar SIN  calcular, porque no tienen suficiente historia
<br> Estra primera pasada lleva 55 minutos !

In [ ]:
# solo los productos que faltan

productos = tb_final.filter(
    pl.col("tn").is_null()
).select("product_id" ).get_column("product_id").to_list()


# Es fundamental que tb_ventas este ORDENADO
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

for producto in productos:

  print(producto, end=' ')
  tn_values = tb_ventas.filter(pl.col("product_id") == producto).select(["tn"])

  try:
    modelo = auto_arima(
      tn_values,
      seasonal=True,
      m=12, # estacinalidad cada 12
      stepwise=True,
      suppress_warnings=True,
      error_action="ignore",
      max_p=3, max_q=3,
      max_P=2, max_Q=2,
      random=True,
      random_state=PARAM['semilla_primigenia'],
      n_fits=10 # cantidad de iteraciones de busqueda AUTO
    )

    # predigo el periodo+1 y periodo+2
    forecast = modelo.predict(n_periods=2)
    mesmasdos = forecast[1]  # el segundo elemento del vector

    tb_final = tb_final.with_columns(
       pl.when(pl.col("product_id") == producto)
      .then(mesmasdos if mesmasdos >=0  else 0 )
      .otherwise(pl.col("tn")).alias("tn")
    )

  except Exception as e:
    print(f"  ERROR for  {producto} ")

20001 20002 20003 20004 20005 20006 

In [ ]:
# cuento cuantos registros NO puedieron calcularse
# en mi corrida esto da ~ 210

tb_final["tn"].is_null().sum()

In [ ]:
arch_tb_final

In [ ]:
# grabo la historia

tb_final.write_csv(arch_tb_final)

## 1.4  Segunda Pasada AutoARIMA

Ahora NO le indico estacionalidad de 12 periodos

Me quedaron prodcutos que NO puede construir el modelo ARIMA debido a no tener suficiente historia para la estacionalidad de 12 meses.
<br> Ahora  quito la condicion de estacionalidad y me encomiendo a la diosa del forecasting

In [ ]:
# ahora SIN  estacionalidad

# solo los productos que faltan
productos = tb_final.filter(
    pl.col("tn").is_null()
).select("product_id" ).get_column("product_id").to_list()

# Es fundamental que tb_ventas este ORDENADO
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

for producto in productos:

  print(producto, end=' ')
  tn_values = tb_ventas.filter(pl.col("product_id") == producto).select(["tn"])

  try:
    modelo = auto_arima(
      tn_values,
      seasonal= False,  # Sin estacionalidad
      stepwise=True,
      suppress_warnings=True,
      error_action="ignore",
      max_p=3, max_q=3,
      max_P=2, max_Q=2,
      random=True,
      random_state=PARAM['semilla_primigenia'],
      n_fits=10
    )

    forecast = modelo.predict(n_periods=2)
    mesmasdos = forecast[1]

    tb_final = tb_final.with_columns(
       pl.when(pl.col("product_id") == producto)
      .then(mesmasdos if mesmasdos >=0  else 0)
      .otherwise(pl.col("tn")).alias("tn")
    )

  except Exception as e:
    print(f"  ERROR for  {producto} ")

In [ ]:
# cuento cuantos registros NO puedieron calcularse aun en este segundo intento
# a mi me quedaron sin calcular  CERO
tb_final["tn"].is_null().sum()

In [ ]:
# vuelvo a grabar
tb_final.write_csv(arch_tb_final)

## 1.5 Submit a Kaggle

In [ ]:
os.getcwd()

In [ ]:
# Submit a Kaggle
if not empiojar:
  archivo= "autoARIMA.csv"
  mensaje= "autoARIMA DOS fases"
else:
  archivo= f"autoARIMA_empiojado_{empiojar_ruido}.csv"
  mensaje= "autoARIMA logEMPIOJADO al " + str(empiojar_ruido) + " ,dos fases"

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )